In [1]:
import numpy as np 
import matplotlib.pyplot as plt

In [2]:
from pyfluids import Fluid, FluidsList, Input

import numpy as np 
T = 90 #K
T_C = T - 273.15 #Celsius

ethane = Fluid(FluidsList.Ethane)
# Set state for ethane
Temperature = 180
T_celsius = Temperature - 273.15  # Convert Kelvin to Celsius
ethane = ethane.with_state(
    Input.temperature(T_celsius),  # Temperature in Kelvin
    Input.pressure(101325)   # Pressure in Pascals (≈1 atm)
)

# Calculate properties
k_ethane = ethane.conductivity
rho_ethane = ethane.density
cp_ethane = ethane.specific_heat
surface_tension = 0.025 # N/m for ethane in air 
dynamic_viscosity = ethane.dynamic_viscosity
print(f"Ethane properties, 1 atm:")
print(f"Thermal conductivity: {k_ethane:.4f} W/(m*K)")
print(f"Density: {rho_ethane:.4f} kg/m^3")
print(f"Specific heat: {cp_ethane:.4f} J/(kg*K)")
print(f"Surface tension: {surface_tension:.4f} N/m")
print(f"Dynamic viscosity: {dynamic_viscosity:.4e} Pa*s")
thermal_diffusivity_ethane = k_ethane / (rho_ethane * cp_ethane)
print(f"Thermal diffusivity: {thermal_diffusivity_ethane:.4e} m^2/s")


radius_1 = 250e-6 # m
radius_2 = 1.7e-3 # m
velocity = 1# m/s
density = rho_ethane # kg/m^3



Ethane properties, 1 atm:
Thermal conductivity: 0.1713 W/(m*K)
Density: 549.5297 kg/m^3
Specific heat: 2421.3569 J/(kg*K)
Surface tension: 0.0250 N/m
Dynamic viscosity: 1.7615e-04 Pa*s
Thermal diffusivity: 1.2877e-07 m^2/s


In [3]:
dynamic_pressure = density * velocity**2 
print(f"Dynamic pressure: {dynamic_pressure:.2f} Pa")
laplace_pressure_1 = surface_tension * (radius_2 / radius_1**2)
laplace_pressure_2 = surface_tension * (radius_1 / radius_2**2)
print(f"Laplace pressure for radius {radius_1*1e6:.2f} um: {laplace_pressure_1:.2f} Pa")
print(f"Laplace pressure for radius {radius_2*1e6:.2f} um: {laplace_pressure_2:.2f} Pa")


Dynamic pressure: 549.53 Pa
Laplace pressure for radius 250.00 um: 680.00 Pa
Laplace pressure for radius 1700.00 um: 2.16 Pa


In [4]:
# Rayleigh Plateu instability calculations (Chandrasekhar 1961). Aristoff https://thales.mit.edu/bush/wp-content/uploads/2012/09/aristoff-splash.pdf
r = 1.50e-3 # m 
time_pinch_rp = 1.2 * np.sqrt(rho_ethane * r**3 / surface_tension)
print(f"Time for Rayleigh Plateau instability: {time_pinch_rp*1000:.2f} ms")

Time for Rayleigh Plateau instability: 10.34 ms


In [5]:
capillary_length = np.sqrt(surface_tension / (density * 9.81))
print(f"Capillary length: {capillary_length*1000:.2f} mm")

Capillary length: 2.15 mm


In [6]:
rho_solid = 8960 # kg/m^3, density of copper
density_ratio_solid_liquid = rho_solid / rho_ethane
print(f"Density ratio (ethane/copper): {density_ratio_solid_liquid:.2f}")



Density ratio (ethane/copper): 16.30


In [7]:
reynolds_number = lambda rho, velocity, length, viscosity: (rho * velocity * length) / viscosity
capillary_number = lambda surface_tension, velocity, viscosity: (viscosity * velocity) / surface_tension
bond_number = lambda rho, gravity, length, surface_tension: (rho * gravity * length**2) / surface_tension
weber_number = lambda rho, velocity, length, surface_tension: (rho * velocity**2 * length) / surface_tension
froude_number = lambda velocity, length, gravity: (velocity**2) / (length * gravity)
froude_number_root = lambda velocity, length, gravity: np.sqrt(velocity**2 / (length * gravity))
ohnesorge_number = lambda viscosity, rho, surface_tension, length: viscosity / np.sqrt(rho * surface_tension * length)
grashoff_number = lambda rho, gravity, length, viscosity, thermal_diffusivity: (rho * gravity * length**3) / (viscosity**2 * thermal_diffusivity)
capillary_length = lambda surface_tension, rho, gravity: np.sqrt(surface_tension / (rho * gravity))

In [8]:
diameter = 3.5e-3 # m
thickness = 300e-6 # m
weber_number_diameter = weber_number(rho_ethane, velocity, diameter, surface_tension)
print(f"Weber number for diameter {diameter*1e3:.2f} mm: {weber_number_diameter:.2f}")
weber_number_thickness = weber_number(rho_ethane, velocity, thickness, surface_tension)
print(f"Weber number for thickness {thickness*1e3:.2f} mm: {weber_number_thickness:.2f}")

weber_number_density_ratio_diameter = weber_number_diameter * density_ratio_solid_liquid
print(f"Weber number for diameter {diameter*1e3:.2f} mm with density ratio: {weber_number_density_ratio_diameter:.2f}")
weber_number_density_ratio_thickness = weber_number_thickness * density_ratio_solid_liquid
print(f"Weber number for thickness {thickness*1e3:.2f} mm with density ratio: {weber_number_density_ratio_thickness:.2f}")

Weber number for diameter 3.50 mm: 76.93
Weber number for thickness 0.30 mm: 6.59
Weber number for diameter 3.50 mm with density ratio: 1254.40
Weber number for thickness 0.30 mm with density ratio: 107.52


In [11]:
# check whether the hydrostatic force is large enough to overcome capillary force to push through a mesh
mesh_height = 25e-6
mesh_width = 100e-6
depth = 1.5e-3 
hydrostatic_pressure = rho_ethane * 9.81 * depth
curvature_pressure = surface_tension * mesh_height / (mesh_width**2)
print(f"Hydrostatic pressure: {hydrostatic_pressure:.2f} Pa")
print(f"Curvature pressure: {curvature_pressure:.2f} Pa")

Hydrostatic pressure: 8.09 Pa
Curvature pressure: 62.50 Pa
